In [ ]:
!pip install datasets transformers torch
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

In [ ]:
# 토크나이저
dataset = load_dataset("bentrevett/multi30k")
MODEL_NAME = "Helsinki-NLP/opus-mt-de-en"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

...

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
print(tokenizer)
'''
# 2. 필터링 함수 정의 (조건을 만족하면 True, 제외하려면 False 반환)
def filter_fn(example):
    # 예시 1: 텍스트가 비어있지 않은 경우만 포함
    has_text = len(example["src_text"].strip()) > 0 and len(example["tgt_text"].strip()) > 0

    # 예시 2: 너무 짧거나 긴 문장 제거 (단어 수 기준)
    length_valid = 5 <= len(example["src_text"].split()) <= 512

    return has_text and length_valid

# 3. 데이터셋에 filter 적용
filtered_dataset = dataset.filter(filter_fn)
'''

'''
preprocess_fn 안에 해당 코드를 넣을 수도 있다.

    prompts = []
    for messages in examples["messages"]:
        # 토큰화 직전에 템플릿을 적용하여 모델 양식에 맞는 하나의 '문자열'로 만듭니다.
        templated_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # 학습 시에는 보통 False(또는 생략), 추론(Inference) 시에 True로 설정합니다.
        )
        prompts.append(templated_prompt)

'''


MarianTokenizer(name_or_path='Helsinki-NLP/opus-mt-de-en', vocab_size=58101, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	58100: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


'\npreprocess_fn 안에 해당 코드를 넣을 수도 있다.\n\n    prompts = []\n    for messages in examples["messages"]:\n        # 토큰화 직전에 템플릿을 적용하여 모델 양식에 맞는 하나의 \'문자열\'로 만듭니다.\n        templated_prompt = tokenizer.apply_chat_template(\n            messages,\n            tokenize=False,\n            add_generation_prompt=True  # 학습 시에는 보통 False(또는 생략), 추론(Inference) 시에 True로 설정합니다.\n        )\n        prompts.append(templated_prompt)\n\n'

In [ ]:
dataset['train'][0]
tokenizer.eos_token

'</s>'

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [ ]:
MAX_LENGTH = 120

def preprocess_fn(example):
    src_text = example['en']
    tgt_text = example['de']

    # 1. 텍스트 문자열 양끝에 <bos>와 <eos> 토큰 문자열을 추가
    # tokenizer에 bos_token / eos_token이 설정되어 있을 경우 사용합니다.
    src_texts = [tokenizer.eos_token + text + tokenizer.eos_token for text in example['en']]
    tgt_texts = [tokenizer.eos_token + text + tokenizer.eos_token for text in example['de']]

    # 2. Tokenization 진행
    model_input = tokenizer(
        src_texts,
        max_length=MAX_LENGTH,
        truncation=True
    )

    label = tokenizer(
        tgt_texts,
        max_length=MAX_LENGTH,
        truncation=True
    )

    model_input['labels'] = label['input_ids']
    return model_input

tokenized_dataset = dataset.map(
    preprocess_fn,
    batched = True,
    remove_columns = dataset['train'].column_names
)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
datacollator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    padding = True,
    return_tensors = 'pt',
)

train_dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size = 32,
    shuffle = True,
    collate_fn = datacollator
)

val_dataloader = DataLoader(
    tokenized_dataset['validation'],
    batch_size = 32,
    shuffle = True,
    collate_fn = datacollator
)

In [ ]:
for batch in train_dataloader:
  print(batch)
  print(batch['input_ids'].shape)
  print(batch['attention_mask'].shape)
  print(batch['labels'].shape)

  break

  # 0은 패딩 토큰 (어텐션에서 무시되는 값)
  # -100 은 로스함수에서 무시되는 값
  # {'input_ids': , 'attention_mask':,  'labels':}
  # input : (32, 36)
  # 가장 긴 문장 길이인 36에 맞춰서 패딩(Padding) 처리합니다.


{'input_ids': tensor([[    0,    93,   878,  ..., 58100, 58100, 58100],
        [    0,    93,   175,  ..., 58100, 58100, 58100],
        [    0,    36, 13464,  ..., 58100, 58100, 58100],
        ...,
        [    0,  1002,   175,  ..., 58100, 58100, 58100],
        [    0,    93,   175,  ..., 58100, 58100, 58100],
        [    0,    93, 32672,  ..., 58100, 58100, 58100]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[   0,  525, 1354,  ..., -100, -100, -100],
        [   0,  246, 1155,  ..., -100, -100, -100],
        [   0,   55,  148,  ..., -100, -100, -100],
        ...,
        [   0,  246, 1155,  ..., -100, -100, -100],
        [   0,  246, 1155,  ..., -100, -100, -100],
        [   0,  246, 5324,  ..., -100, -100, -100]])}
torch.Size([32, 47])
torch.Size([32, 47])
torch.S

In [ ]:
# 정리 : 데이터 콜레이터의 역할
# 1. 딕셔너리나 리스트로 들어온 경우 데이터를 텐서로 변환
# 2. 동적 패딩
# 3. 손실함수의 예외처리 (-100 마스킹)
# 4. 태스크별 동적 데이터 가공
# 4.1 MLM(Masked Language Model)
'''
실시간으로 토큰의 15%를 무작위로 [MASK] 토큰으로 교체하고, 정답 라벨을 생성함
'''
# 4.2 Seq2Seq
'''
라벨을 자동 추출하고 오른쪽으로 Shift작업을 해준다.
(decoder_input_ids 생성)
'''
# 4.3 Decoder
'''
Shift작업은 모델 내부 코드에서 진행해주기에 따로 하지 않는다.
라벨 생성

'''

# 정리 : 토크나이저 사용시 고려할 점
# 1. vocab_size (한국어가 있는지, 3~10만인지)
# 2. 사전학습 모델과의 일치성
# 3. 특수토큰 선정 여부 (pad, imstart)
# 4. 패딩 방향
# 5. Fast Tokenizer (is_fast)
# 6. 콜레이터 사용시 적정한 모델 사용인지

In [ ]:
# Training
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
import torch.optim as optim
from transformers import (
    AutoTokenizer,
    AutoConfig,
    EncoderDecoderModel,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup )

In [ ]:
config = AutoConfig.from_pretrained("Helsinki-NLP/opus-mt-de-en")
print(config)

MarianConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "swish",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "MarianMTModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 512,
  "decoder_attention_heads": 8,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 58100,
  "decoder_vocab_size": 58101,
  "dropout": 0.1,
  "encoder_attention_heads": 8,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 0,
  "forced_eos_token_id": 0,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 512,
  "model_type": "marian",
  "normalize_before": false,
  "n

In [ ]:
# 토크나이저 config에 맞춰 모델 config도 맞춰주어야 함.
# 1. vocan_size = 58101
# 2.
'''
pad_token_id: 배치(Batch) 처리 시 길이를 맞추기 위한 패딩 토큰 ID.

bos_token_id: 문장/시퀀스의 시작(Beginning of Sequence) 토큰 ID.

eos_token_id: 문장/시퀀스의 끝(End of Sequence) 토큰 ID.

unk_token_id: 사전외 단어(Unknown) 토큰 ID.
'''
# 3.
'''
Tokenizer 설정: model_max_length 또는 max_len
Model Config 설정: max_position_embeddings, seq_len, 또는 n_positions
'''
# 4.
'''
3. 채팅/인스트럭션 템플릿 (Chat Template)
최신 LLM(Instruction/Chat 모델)을 다룰 때 매우 중요한 요소입니다.

chat_template: tokenizer_config.json 내에 Jinja2 템플릿 형태로 저장되는 시스템/유저/어시스턴트 프롬프트 포맷입니다.

특수 역할 토큰: <|im_start|>, <|im_end|> 또는 <|user|>, <|assistant|> 등 템플릿에서 사용하는 특수 토큰이 토크나이저의 added_tokens에 올바르게 등록되어 있는지, 해당 토큰들의 ID가 모델 학습 시 사용된 Special Token들과 일치하는지 확인해야 합니다.

4. Special Token 관련 동작 플래그 (Add Special Tokens)
토크나이징 시 자동으로 BOS/EOS 토큰을 붙일지 여부에 대한 설정입니다.

add_bos_token / add_eos_token: 토크나이저 설정에서 이 값이 True로 되어 있는데, 프롬프트 작성 코드나 학습 루프에서 수동으로 BOS/EOS를 또 추가하면 <bos><bos> Hello ...처럼 중복 입력이 들어가 모델 성능이 파괴됩니다.
'''

In [ ]:
PAD_IDX = tokenizer.pad_token_id
PAD_IDX

58100

In [ ]:
# {'input_ids': , 'attention_mask':,  'labels':}
# input : (32, 36) = (batch_size, seq_len)
# 가장 긴 문장 길이인 36에 맞춰서 패딩(Padding) 처리합니다.

# 트랜스포머의 max_len은 512이나, 토크나이저가 120이므로 얘도 120으로 감

# 입력 (seq_len, batch_size, emb_size)
# 출력 (seq_len, batch_size, emb_size)
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len, dropout = 0.1):
    super().__init__()
    self.dropout = nn.Dropout(p = dropout)
    position = torch.arange(max_len).unsqueeze(1)
    div_term = torch.exp(
                          torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model)
                        )
    pe = torch.zeros(max_len, 1, d_model)
    pe[:, 0, 0::2] = torch.sin(position * div_term)
    pe[:, 0, 1::2] = torch.cos(position * div_term)
    self.register_buffer("pe", pe)

  def forward(self, x):
    x = x + self.pe[: x.size(0)]
    return self.dropout(x)


# 입력 (seq_len, batch_size)
# 출력 (seq_len, batch_size, emb_size)
class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, emb_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, emb_size)
    self.emb_size = emb_size

  def forward(self, tokens):
    return self.embedding(tokens.long())*math.sqrt(self.emb_size)
    # 64비트 인티저 변환 + 포지셔널 임베딩 고려하여 스케일링

class Seq2SeqTransformer(nn.Module):
  def __init__(
      self,
      num_encoder_layers, # 6
      num_decoder_layers, # 6
      emb_size,# 512
      max_len,# 120
      nhead, # 8
      src_vocab_size, # 58101
      tgt_vocab_size, # 58101
      dim_feedforward,# 2048
      dropout = 0.1,
      pad_idx = 1
  ):
    super().__init__()
    self.pad_idx = pad_idx
    self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
    self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
    self.positional_encoding = PositionalEncoding(
        d_model = emb_size, max_len = max_len, dropout = dropout
    )
    self.transformer = nn.Transformer(
        d_model = emb_size,
        nhead = nhead,
        num_encoder_layers = num_encoder_layers,
        num_decoder_layers = num_decoder_layers,
        dim_feedforward = dim_feedforward,
        dropout = dropout,
        activation = torch.nn.functional.relu,
        layer_norm_eps = 1e-05
    )
    self.generator = nn.Linear(emb_size, tgt_vocab_size)

  def forward(
      self,
      src, # Tensor (seq_len, batch)
      trg, # Tensor (seq_len, batch)
      tgt_mask = None, # Tensor (seq_len, batch)
      src_mask = None, # 인코더의 마스크[소스 시퀀스, 시퀀스]
      src_padding_mask = None, # Tensor (seq_len, batch) # 디코더가 인코더 결과를 볼 때 쓰는 패딩 마스크
      tgt_padding_mask = None, # Tensor (seq_len, batch)
      memory_key_padding_mask = None, # 트랜스포머(Transformer) 모델에서 인코더(Encoder)의 출력 데이터(Memory) 중 패딩(Padding)된 부분을 디코더(Decoder)가 참조하지 못하도록 가리는(Masking) 행렬
  ):
    if src_padding_mask is None:
      src_padding_mask = (src == self.pad_idx)
      # 입력 텐서 src 안에 있는 숫자(토큰 ID)들 중 어떤 것이 패딩 토큰(<pad>)인지 찾아내어 True / False 형태의 불리언(Boolean) 지도(Mask)를 만드는 코드입니다.

    if tgt_padding_mask is None:
      tgt_padding_mask = (trg == self.pad_idx)

    if memory_key_padding_mask is None:
      # memory_key_padding_mask는 src_padding_mask와 동일하게 설정
      memory_key_padding_mask = src_padding_mask

   # 2. tgt_mask(미래 단어 컨닝 방지 마스크)가 없으면 내부에서 자동 생성
    if tgt_mask is None:
      tgt_len = trg.size(0)

    # Causal Mask (미래 단어 가림 마스크) 생성
      tgt_mask = torch.triu(
          torch.ones((tgt_len, tgt_len), device=trg.device, dtype=torch.bool),
          diagonal=1
      )

    src_emb = self.positional_encoding(self.src_tok_emb(src))
    tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
    outs = self.transformer(
        src = src_emb,
        tgt = tgt_emb,
        src_mask = src_mask,
        tgt_mask = tgt_mask,
        memory_mask = None,
        src_key_padding_mask = src_padding_mask,
        tgt_key_padding_mask = tgt_padding_mask,
        memory_key_padding_mask = memory_key_padding_mask,
    )
    return self.generator(outs)

  def encode(self, src, src_mask):
    return self.transformer.encoder(
        self.positional_encoding(self.src_tok_emb(src)), src_mask
      )

  def decode(self, tgt, memory, tgt_mask):
    return self.transformer.decoder(
        self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
      )
